# Project 2: Structured Data Extraction with a Large Language Model API

## Overview

In this project we use the **OpenAI API** to extract structured information from unstructured text — without any training data. Rather than teaching a model our task (as in Project 1), we *describe* the task in plain language and ask a frontier model to do it.

This is called **zero-shot prompting**: the model performs the task based on instructions alone, drawing on knowledge it acquired during pre-training on vast amounts of text.

**What is an API?**  
An API (Application Programming Interface) is a way to send requests to a remote service and receive responses. The OpenAI API lets us send text (and images) to GPT-4o and receive the model's output — programmatically, in our Python code. We pay per request based on the number of tokens processed.

## Task: Extracting Information from UK Parliamentary Bills

We will work with a dataset of successful **Private Members' Bills** from the UK House of Commons (1983–present). Each bill has a title like:

> *"Taxis and Private Hire Vehicles (Safeguarding and Road Safety) Bill 2021-22"*

From each title, we will ask GPT to extract:
- `policy_area` — the broad policy domain (e.g., *Transport*, *Criminal Justice*)
- `affected_group` — who the bill primarily targets (e.g., *taxi drivers*, *children*)
- `geographic_scope` — does it apply to England, Scotland, UK-wide?
- `bill_summary` — a one-sentence plain-English explanation

This mimics a task many social scientists perform manually at enormous cost: reading documents and extracting structured variables for quantitative analysis.

## Learning objectives
1. Set up and use the OpenAI Python SDK
2. Write effective prompts that return structured (JSON) output
3. Process a corpus of documents with the API
4. Validate LLM-based extractions against a manual gold standard


## Before You Start: Getting an API Key

To use the OpenAI API you need an account and API key:

1. Go to [platform.openai.com](https://platform.openai.com) and create an account  
2. Navigate to **API Keys** and create a new secret key (starts with `sk-...`)  
3. Add a small amount of credit (£5–10 is plenty for this workshop)  

**The cost of running this notebook in full is approximately $0.10–$0.30** depending on how many bills you process.

> **Keep your key private.** Do not paste it directly into a shared notebook or commit it to GitHub. Use environment variables or Colab Secrets (see the setup cell below).


## Setup: Install the OpenAI library


In [ ]:
pip install openai pandas matplotlib

## 1. Import Libraries and Set Up the API Client

The `OpenAI` client is our gateway to the API. We pass it our API key and it handles authentication for every request we make.

**How to set your key (choose one method):**
- **Colab:** Store the key in Colab Secrets (the 🔑 icon in the left panel) under the name `OPENAI_API_KEY`, then uncomment the Colab block below
- **Local:** Set an environment variable: `export OPENAI_API_KEY="sk-..."` in your terminal before launching Jupyter
- **Quick test only:** Paste your key directly into `api_key=` (remove it before sharing the notebook)


In [ ]:
import os
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI

# ── Option 1: read from environment variable (recommended for local use) ──────
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Option 2: Google Colab Secrets ───────────────────────────────────────────
# from google.colab import userdata
# client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# ── Option 3: paste directly (for quick testing only — do not share!) ─────────
# client = OpenAI(api_key="sk-YOUR-KEY-HERE")

print("Client created. Sending a test request...")

# Quick test to confirm the key is working
test = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with exactly: API connection successful."}],
    max_tokens=20,
)
print(test.choices[0].message.content)


## 2. Load and Explore the Dataset

Our dataset contains **404 successful Private Members' Bills** from the UK House of Commons, spanning 1983 to the present day. Private Members' Bills are legislation introduced by backbench MPs (rather than the government) — they often reflect niche policy concerns or are used to test political waters.

The key column we will use is `'Title of bill as presented'` — this is the raw bill title from which we will extract structured information.


In [ ]:
# Load the dataset
# Adjust the path if needed (on Colab, upload the file or clone the repo)
bills = pd.read_csv('../data/hoc_private_member_bills.csv', header=2, skiprows=[3])

print(f"Dataset: {bills.shape[0]} bills, {bills.shape[1]} columns")
print(f"Columns: {bills.columns.tolist()}")
bills.head()


In [ ]:
# Let's look at the column we'll use most: the bill title
print("Sample bill titles:")
print("-" * 60)
for title in bills['Title of bill as presented'].sample(10, random_state=42).values:
    print(f"  • {title}")


In [ ]:
# Bills over time — how has the volume changed?
bills['Parliament'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Successful Private Members\' Bills by Parliament')
plt.xlabel('Parliament')
plt.ylabel('Number of Bills')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 3. Making API Calls: The Basics

The OpenAI chat API works with a **conversation format**. Each request contains a list of messages:

| Role | Purpose |
|------|---------|
| `system` | Persistent instructions that define the model's behaviour and persona |
| `user` | The actual input (equivalent to what you type in ChatGPT) |
| `assistant` | The model's previous replies (used in multi-turn conversations) |

For our extraction task, the **system message** will define the extraction schema and output format. The **user message** will contain the bill title we want to analyse.

Let's start with a simple example before building our full extraction pipeline.


In [ ]:
# A simple API call to understand the basic structure
response = client.chat.completions.create(
    model="gpt-4o-mini",           # fast and cost-effective; use "gpt-4o" for best quality
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant specialising in UK parliamentary procedure."
        },
        {
            "role": "user",
            "content": "In one sentence, what is a Private Member's Bill in the UK Parliament?"
        }
    ],
    temperature=0,                 # 0 = deterministic output; good for structured tasks
    max_tokens=150,
)

# The model's reply is in choices[0].message.content
print(response.choices[0].message.content)


## 4. Designing the Extraction Prompt

Good prompts for structured extraction tasks share a few properties:

1. **Clear role definition**: Tell the model what kind of expert it should act as  
2. **Explicit output format**: Specify that you want JSON, and describe every field  
3. **Field definitions**: Define each field precisely, including acceptable values where relevant  
4. **Constrained outputs**: For categorical fields, provide the list of allowed values — this prevents the model inventing new categories  
5. **Example (optional)**: A single worked example dramatically improves consistency

We will instruct the model to return a **JSON object** with four fields. JSON (JavaScript Object Notation) is a structured text format that is easy to parse back into Python dictionaries.

### Our extraction schema

```json
{
  "policy_area": "string — broad policy domain, e.g. Transport, Health, Criminal Justice",
  "affected_group": "string — who the bill primarily targets, e.g. 'children', 'NHS workers', 'tenants'",
  "geographic_scope": "one of: England, Scotland, Wales, Northern Ireland, UK-wide, Other",
  "bill_summary": "string — one plain-English sentence explaining what the bill does"
}
```


In [ ]:
# Define the system prompt — this stays constant for every bill we process
SYSTEM_PROMPT = """You are an expert in UK parliamentary legislation and public policy.

Your task is to analyse the title of a UK House of Commons Private Member's Bill and extract structured information.

Return ONLY a valid JSON object with exactly these four fields:

{
    "policy_area": "The broad policy domain (e.g. Transport, Health, Education, Criminal Justice, Environment, Housing, Labour Rights, Animal Welfare, Family Law, Finance, Other)",
    "affected_group": "The primary group or sector the bill targets (e.g. 'children', 'NHS staff', 'taxi drivers', 'tenants', 'small businesses', 'animals', 'the general public')",
    "geographic_scope": "One of: England, Scotland, Wales, Northern Ireland, UK-wide, Other",
    "bill_summary": "A single plain-English sentence explaining what the bill does or would change"
}

Rules:
- Output only the JSON object, nothing else (no markdown, no explanation)
- If geographic scope is not specified in the title, infer it from context (Scottish bills usually specify Scotland)
- Keep bill_summary concise and factual
"""

def extract_bill_info(bill_title: str) -> dict:
    """
    Call the OpenAI API to extract structured information from a bill title.
    Returns a Python dictionary, or None if parsing fails.
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Bill title: {bill_title}"},
            ],
            temperature=0,        # deterministic output for research reproducibility
            max_tokens=200,
            response_format={"type": "json_object"},  # force JSON output
        )
        raw = response.choices[0].message.content
        return json.loads(raw)    # parse JSON string → Python dict
    except (json.JSONDecodeError, Exception) as e:
        print(f"  Error processing '{bill_title}': {e}")
        return None

print("Extraction function defined.")


### Testing on a single bill

Before processing the whole corpus, always test your prompt on a few examples. This helps you catch problems with the prompt design before spending money on a large batch.


In [ ]:
# Test on a single bill title
test_title = "Taxis and Private Hire Vehicles (Safeguarding and Road Safety) Bill 2021-22"

result = extract_bill_info(test_title)

print(f"Input:  {test_title}")
print()
print("Extracted information:")
for key, value in result.items():
    print(f"  {key:<20}: {value}")


In [ ]:
# Test on a few more bills to check consistency
test_bills = [
    "Co-operatives, Mutuals and Friendly Societies Bill 2022-23",
    "Protection of Animals (Scotland) Bill 1992/93",
    "Access to Medical Treatments (Innovation) Bill 2015-16",
    "Sunday Working (Scotland) Bill 2002/03",
]

print("Testing extraction on multiple bills:")
print("=" * 70)
for title in test_bills:
    result = extract_bill_info(title)
    print(f"\nBill: {title}")
    if result:
        for k, v in result.items():
            print(f"  {k:<20}: {v}")


## 5. Extracting Across the Full Dataset

Now we run the extraction over all bills in our dataset. A few practical notes:

**Cost estimate:**  
Each bill title is short (~10–20 words). With `gpt-4o-mini` pricing (~$0.15 per million input tokens), processing all 404 bills costs approximately **$0.05**. The cell below processes the full dataset — if you want to test with a subset first, change `n_bills`.

**Rate limits:**  
OpenAI limits the number of requests per minute. We add a small delay between requests to stay within the free-tier limits. If you see a `RateLimitError`, increase the `time.sleep()` delay.

**Error handling:**  
We wrap each API call in a try/except block and store `None` for any failed requests. Real research pipelines should retry failed requests automatically — we keep it simple here.


In [ ]:
# How many bills to process (set to None to process all 404)
n_bills = None   # change to e.g. 50 to run a quick test

df_to_process = bills if n_bills is None else bills.head(n_bills)

results = []

print(f"Processing {len(df_to_process)} bills...")
print("=" * 60)

for i, row in df_to_process.iterrows():
    title = row['Title of bill as presented']

    extracted = extract_bill_info(title)

    if extracted:
        extracted['bill_title'] = title
        extracted['presented_by'] = row.get('Presented by', '')
        extracted['party'] = row.get("Member's party (1)", '')
        extracted['parliament'] = row.get('Parliament', '')
        results.append(extracted)

    # Progress update every 25 bills
    if (len(results)) % 25 == 0 and len(results) > 0:
        print(f"  {len(results)} bills processed...")

    time.sleep(0.1)   # small delay to respect rate limits

print(f"\nFinished. Successfully extracted: {len(results)} / {len(df_to_process)} bills.")


In [ ]:
# Convert results to a DataFrame for analysis
results_df = pd.DataFrame(results)

# Reorder columns for readability
cols = ['bill_title', 'policy_area', 'affected_group', 'geographic_scope', 'bill_summary', 'presented_by', 'party', 'parliament']
results_df = results_df[cols]

print(f"Results shape: {results_df.shape}")
results_df.head(10)


## 6. Analysing the Extracted Data

One of the main advantages of LLM-based extraction is that it converts unstructured text into structured variables that we can analyse quantitatively. Here we do some simple descriptive analysis on the extracted fields.


In [ ]:
# What are the most common policy areas?
policy_counts = results_df['policy_area'].value_counts()

print("Most common policy areas in successful Private Members' Bills:")
print(policy_counts.head(15).to_string())

policy_counts.head(12).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title("Policy Areas of Successful Private Members' Bills")
plt.xlabel('Number of Bills')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Geographic scope distribution
scope_counts = results_df['geographic_scope'].value_counts()
print("Geographic scope of bills:")
print(scope_counts)

scope_counts.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Geographic Scope of Bills')
plt.xlabel('Scope')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Which parties introduced bills in which policy areas?
# Cross-tabulate party by policy area (top 6 policy areas)
top_areas = results_df['policy_area'].value_counts().head(6).index
party_area = results_df[results_df['policy_area'].isin(top_areas)].copy()

pd.crosstab(party_area['party'], party_area['policy_area'])


## 7. Validating the Extractions

Automated extraction is only useful if it is accurate. Validation is a critical step that is often skipped — do not skip it in your own research.

A simple validation approach:
1. Randomly sample 20–30 bills from the results
2. Manually code the same bills yourself (or have a research assistant do it)
3. Compare your codes to the model's codes
4. Compute **agreement** (simple %) and **Cohen's Kappa** (agreement adjusted for chance)

Below we demonstrate this for a small sample of 10 bills, comparing the model's `policy_area` to a hand-coded gold standard. In real research, use a larger sample (typically ≥50 items per category).

> **Tip:** Disagreements are informative. They reveal where the model is inconsistent, where your categories are ambiguous, or where the task is genuinely difficult — all useful for improving your pipeline.


In [ ]:
# --- Manual validation example ---
# Below are 10 bill titles with hand-coded policy areas.
# Compare these to what the model extracted.

# For this exercise, code these yourself first, then run the cell.
validation_titles = [
    "Anatomy Bill 1983/84",
    "Corneal Tissue Bill 1985/86",
    "Motor Vehicles (Wearing of Rear Seat Belts by Children) Bill 1987/88",
    "Solicitors (Scotland) Bill 1987/88",
    "Child Abduction Bill 1983/84",
    "Cycle Tracks Bill 1983/84",
    "Agriculture (Amendment) Bill 1983/84",
    "Forestry Bill 1985/86",
    "Access to Medical Treatments (Innovation) Bill 2015-16",
    "Betting, Gaming and Lotteries (Amendment) Bill 1983/84",
]

# Hand-coded gold standard labels (edit these to reflect your own coding)
human_codes = [
    "Health",               # Anatomy Bill
    "Health",               # Corneal Tissue Bill
    "Transport",            # Motor Vehicles / Seat Belts
    "Law / Legal Profession",  # Solicitors (Scotland)
    "Criminal Justice",     # Child Abduction
    "Transport",            # Cycle Tracks
    "Agriculture",          # Agriculture (Amendment)
    "Environment",          # Forestry
    "Health",               # Access to Medical Treatments
    "Gambling / Leisure",   # Betting, Gaming and Lotteries
]

# Get model predictions for the same bills
model_codes = []
for title in validation_titles:
    result = extract_bill_info(title)
    model_codes.append(result['policy_area'] if result else "ERROR")
    time.sleep(0.1)

# Compare
val_df = pd.DataFrame({
    'bill': validation_titles,
    'human': human_codes,
    'model': model_codes,
})
val_df['match'] = val_df['human'].str.lower() == val_df['model'].str.lower()

print("Validation Results:")
print("=" * 90)
print(val_df[['bill', 'human', 'model', 'match']].to_string(index=False))
print(f"\nSimple agreement: {val_df['match'].mean():.0%}")


### Interpreting disagreements

When the model disagrees with your hand codes, ask:
- Is the model's label actually reasonable? (If yes, your categories may be ambiguous)
- Is the model systematically wrong for a type of bill? (If yes, revise the prompt)
- Is this bill genuinely ambiguous? (If yes, consider a multi-label scheme or an "ambiguous" category)

In a full research workflow, you would iterate on the prompt, re-run the validation, and report the final agreement statistics alongside your results.


In [ ]:
# Save the extracted data to a CSV for further analysis
output_path = '../data/hoc_bills_extracted.csv'
results_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")
print(f"Shape: {results_df.shape}")


## 8. Key Takeaways & Research Considerations

### What we did
1. Set up the OpenAI API and made our first programmatic call
2. Designed a structured extraction prompt with a defined JSON schema
3. Processed a dataset of 404 parliamentary bills
4. Analysed the extracted variables descriptively
5. Validated a sample of extractions against hand-coded labels

### Strengths of this approach
- **Zero labelled training data required** — works immediately out of the box
- **Flexible** — change the schema without re-training a model
- **Readable output** — the `bill_summary` field gives a human-interpretable result alongside the structured codes
- **Fast** — processing 400 documents takes seconds

### Limitations and cautions
- **Non-determinism**: even at `temperature=0`, GPT may give different outputs across runs. Always check reproducibility for research use.
- **Hallucination**: the model may produce plausible-sounding but incorrect information. Always validate.
- **Black box**: you cannot inspect *why* the model assigned a given label — harder to defend methodologically.
- **Cost and dependency**: you pay per token and depend on a third-party service that can change pricing, terms, or availability.
- **Data privacy**: do not send sensitive, confidential, or personally identifiable data to commercial APIs without appropriate consent and data processing agreements.

### When to use this approach
- Pilot studies and exploratory analysis
- Tasks with complex, open-ended categories that are hard to define in advance
- Extraction of multiple fields simultaneously from the same text
- Cases where training data is unavailable
